# Time-dependent RF evaluation, v2 (global feature set from `tabular_data_prep`; one Optuna study on the full-record data)
Same landmark design as ProtoPregNet: the same fixed 8-1-1 person split + labels (`protopregnet_landmark_cohort`), the same eligibility table, weeks 0-37, open-horizon labels,
the same metrics and bootstrap CIs. What is new in v2:
- **Features:** the GLOBAL selected set from `tabular_data_prep` (chosen once on the full-record TRAIN rows; no per-landmark selection). Each landmark just censors the stored matrix
  at its cutoff (entries are REMOVED structurally, so a visible week-0 first occurrence stays 'present') and builds the design: presence columns (from the sparse structure) +
  value columns (mean-imputed). Window parameters and the matrix fingerprint are read from `tabular_v2/meta.json`, so nothing can mismatch.
- **Hyperparameters:** `mode=tune` runs ONE Optuna study on the FULL-record data (train rows of all patients, no censoring; objective = validation macro AUROC; the test split is never used);
  its best config is then used, unchanged, for every landmark.
- **Model:** one joint multi-output random forest (the three labels are multi-hot), fit on TRAIN only, as before. `n_estimators` is fixed (a monotone cap), the rest is tuned.

**Widgets (`mode`)**: `summary` (default, safe) | `prep` (cache the v2 matrix on local disk; skipped if the cache matches) | `tune` (the Optuna study; resumable) | `run` (one landmark, `cutoff_week`; idempotent).
**Parallelism on one CPU node:** `tune`: `parallel_trials` trials at once, each using `n_jobs // parallel_trials` threads; `run`: several landmarks as separate job tasks, each using `n_jobs` threads (concurrency x n_jobs <= cores).
Run `tabular_data_prep` first. Results go to `*_v2_*` tables, so earlier results are not mixed in.

In [0]:
import os, time, json, gc, shutil
import numpy as np
import pandas as pd
import scipy.sparse as sp
import sklearn
from sklearn.ensemble import RandomForestClassifier
from joblib import Parallel, delayed
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                             brier_score_loss, balanced_accuracy_score)
import pyspark.sql.functions as sf
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

dbutils.widgets.dropdown("mode", "summary", ["summary", "prep", "tune", "run"])
dbutils.widgets.text("cutoff_week", "20")
dbutils.widgets.text("n_jobs", "8")
dbutils.widgets.text("n_trials", "100")
dbutils.widgets.text("parallel_trials", "4")
dbutils.widgets.dropdown("force", "false", ["false", "true"])
MODE = dbutils.widgets.get("mode")
CUTOFF_WEEK = int(dbutils.widgets.get("cutoff_week"))
N_JOBS = int(dbutils.widgets.get("n_jobs"))
N_TRIALS = int(dbutils.widgets.get("n_trials"))
PARALLEL_TRIALS = int(dbutils.widgets.get("parallel_trials"))
FORCE = dbutils.widgets.get("force") == "true"

PERSON_ID_COL = "person_id"
CLASS_NAMES = ["control", "pe_ghtn", "gdm"]
OUTCOME_NAMES = ["pe_ghtn", "gdm"]
LABEL_COLS = ["y_control", "y_pe_ghtn", "y_gdm"]          # same label definitions as the landmark cohort table


def _pick_local_base():
    """Node-local scratch (matrix cache, Optuna journal): /local_disk0 on Databricks nodes, /tmp otherwise."""
    for base in ("/local_disk0", "/tmp"):
        if os.path.isdir(base) and os.access(base, os.W_OK):
            return base
    return "/tmp"


CFG = {
    # ---- inputs: written by tabular_data_prep, and the ProtoPregNet split / eligibility ----
    "sparse_long_table": "catalog.schema.tabular_v2_sparse_long",
    "person_index_table": "catalog.schema.tabular_v2_person_index",
    "feature_index_table": "catalog.schema.tabular_v2_feature_index",
    "selected_table": "catalog.schema.tabular_v2_selected_features",       # the GLOBAL selected feature set (selected_final = true)
    "tabular_meta_path": "/path/to/exports/tabular_v2/meta.json",   # window parameters + matrix fingerprint
    "split_table": "catalog.schema.protopregnet_landmark_cohort",            # the EXACT ProtoPregNet 8-1-1 split (seed 42) + labels
    "eligibility_table": "catalog.schema.protopregnet_landmark_eligibility",   # (pid_int, cutoff_week)
    "ppn_sizes_table": "catalog.schema.protopregnet_landmark_sizes",          # used only to cross-check eligible counts
    "cache_dir": "/path/to/exports/tabular_v2_cache",          # durable copy (Volumes: sequential writes only); shared by the RF and XGBoost notebooks
    "local_cache_dir": f"{_pick_local_base()}/tabular_v2_cache",                   # the node's local disk: what the tasks actually read

    # ---- outputs (v2 tables, so earlier results are not mixed in) ----
    "metrics_table": "catalog.schema.protopregnet_rf_v2_landmark_metrics", "sizes_table": "catalog.schema.protopregnet_rf_v2_landmark_sizes",
    "model_id_template": "rf_v2_landmark_wk{k:02d}",
    "hpo_dir": "/path/to/exports/rf_v2_hpo", "hpo_local_dir": f"{_pick_local_base()}/rf_v2_hpo",
    "landmarks": list(range(0, 38)),
    "n_jobs": N_JOBS,
    
    # ---- model: parameters that are NOT tuned ----
    "rf_fixed": {"n_estimators": 300, "random_state": 42, "oob_score": False, "verbose": 0},   # more trees never hurt, so this is a cost cap, not a tuned value
    "report_val": True,           # print validation macro AUROC at each landmark (the forest is not fit on validation; it only chose the hyperparameters)

    # ---- Optuna: ONE study on the full-record data; its best config is then used, unchanged, for all landmarks ----
    "hpo": {
        "study_name": "rf_v2_full_data", "n_trials": N_TRIALS, "parallel_trials": PARALLEL_TRIALS,
        "seed": 42,                    # TPE seed; with parallel trials the order trials are proposed in is not reproducible
        "n_startup_trials": 20,        # random trials before TPE starts modelling
        # kind: float | int | cat; 'when' = only suggested if the parent has that value (max_samples only applies with bootstrap)
        "space": {
            "criterion":         {"kind": "cat", "choices": ["gini", "entropy"]},
            "max_depth":         {"kind": "cat", "choices": [None, 8, 12, 16, 24, 32]},
            "min_samples_leaf":  {"kind": "int", "low": 1, "high": 50, "log": True},
            "min_samples_split": {"kind": "int", "low": 2, "high": 40, "log": True},
            "max_features":      {"kind": "float", "low": 0.02, "high": 0.5, "log": True},
            "class_weight":      {"kind": "cat", "choices": [None, "balanced"]},
            "bootstrap":         {"kind": "cat", "choices": [True, False]},
            "max_samples":       {"kind": "float", "low": 0.3, "high": 1.0, "when": ("bootstrap", True)},
        },
    },

    # ---- bootstrap CIs on the test set (identical to the earlier notebooks) ----
    "n_bootstrap": 1000, "bootstrap_chunks": 8, "ci": 0.95, "bootstrap_seed": 42,
    "write_retries": 3,
}
FULL_SPARSE_LONG_TABLE = CFG["sparse_long_table"]
PERSON_INDEX_TABLE = CFG["person_index_table"]
FEATURE_INDEX_TABLE = CFG["feature_index_table"]
print(f"mode={MODE} | cutoff_week={CUTOFF_WEEK} | n_jobs={N_JOBS} | sklearn {sklearn.__version__} | numpy {np.__version__}")

In [0]:
# ===== cache I/O for the v2 matrix: unchanged from the v1 notebooks except save_cache records the matrix fingerprint =====
CACHE_FILES = ["X_data.npy", "X_indices.npy", "X_indptr.npy", "person_index.npy", "feature_names.json", "meta.json"]


def _cp_to_volume(local_path, volume_path):
    """Volumes (FUSE) accept only sequential writes, so files are copied in from local disk instead of being written in place."""
    try:
        dbutils.fs.cp(f"file:{local_path}", volume_path)
    except Exception:
        shutil.copyfile(local_path, volume_path)


def _cp_from_volume(volume_path, local_path):
    try:
        dbutils.fs.cp(volume_path, f"file:{local_path}")
    except Exception:
        shutil.copyfile(volume_path, local_path)


def _mkdir_volume(path):
    try:
        os.makedirs(path, exist_ok=True)
    except OSError:
        dbutils.fs.mkdirs(path)


def read_meta(directory):
    with open(f"{directory}/meta.json") as f:
        return json.load(f)


def save_cache(X, feature_names, person_index, fingerprint):
    """Write the matrix once so every landmark task loads it in seconds instead of re-reading ~680M Delta rows.
    Local disk first (random-access safe), then a sequential copy to the Volume for durability. Plain .npy files (not .npz): no zip seeks."""
    X = X.tocsr()
    loc, tmp = CFG["local_cache_dir"], CFG["local_cache_dir"] + ".tmp"
    shutil.rmtree(tmp, ignore_errors=True); os.makedirs(tmp)
    np.save(f"{tmp}/X_data.npy", X.data); np.save(f"{tmp}/X_indices.npy", X.indices); np.save(f"{tmp}/X_indptr.npy", X.indptr)
    np.save(f"{tmp}/person_index.npy", np.asarray(person_index).astype("int64"))
    with open(f"{tmp}/feature_names.json", "w") as f:
        json.dump(list(feature_names), f)
    with open(f"{tmp}/meta.json", "w") as f:                          # written last
        json.dump({"shape": list(X.shape), "nnz": int(X.nnz), "matrix_version": "tabular_v2", "fingerprint": fingerprint,
                   "created": time.strftime("%Y-%m-%d %H:%M:%S"), "sklearn": sklearn.__version__, "numpy": np.__version__}, f)
    shutil.rmtree(loc, ignore_errors=True); os.rename(tmp, loc)       # local copy becomes visible all at once
    _mkdir_volume(CFG["cache_dir"])
    for fn in CACHE_FILES[:-1] + ["meta.json"]:                       # meta.json last on the Volume too
        _cp_to_volume(f"{loc}/{fn}", f"{CFG['cache_dir']}/{fn}")
    for fn in CACHE_FILES:                                            # verify the copy: a partial copy must fail loudly, not at load time
        a, b = os.path.getsize(f"{loc}/{fn}"), os.path.getsize(f"{CFG['cache_dir']}/{fn}")
        if a != b:
            raise IOError(f"cache copy to the Volume is incomplete: {fn} is {b} bytes there vs {a} bytes locally")
    print(f"cache written and verified: local {loc} and Volume {CFG['cache_dir']}")


def ensure_local_cache():
    """The matrix lives on the node's local disk so concurrent tasks can memory-map ONE copy (shared page cache). If the node has no
    local copy (e.g. the cluster restarted), copy it from the Volume once; a task that loses the race just uses the winner's copy."""
    loc = CFG["local_cache_dir"]
    if os.path.exists(f"{loc}/meta.json"):
        return loc
    if not os.path.exists(f"{CFG['cache_dir']}/meta.json"):
        raise FileNotFoundError(f"no matrix cache in {CFG['cache_dir']} -- run the notebook once with mode=prep first")
    tmp = f"{loc}.tmp_{os.getpid()}"
    shutil.rmtree(tmp, ignore_errors=True); os.makedirs(tmp)
    for fn in CACHE_FILES:
        _cp_from_volume(f"{CFG['cache_dir']}/{fn}", f"{tmp}/{fn}")
    try:
        os.rename(tmp, loc)
    except OSError:
        shutil.rmtree(tmp, ignore_errors=True)                        # another task finished first
    return loc


def load_cache():
    loc = ensure_local_cache()
    with open(f"{loc}/meta.json") as f:
        meta = json.load(f)
    data, indices = np.load(f"{loc}/X_data.npy", mmap_mode="r"), np.load(f"{loc}/X_indices.npy", mmap_mode="r")
    X = sp.csr_matrix((data, indices, np.load(f"{loc}/X_indptr.npy")), shape=tuple(meta["shape"]), copy=False)   # read-only, memory-mapped
    with open(f"{loc}/feature_names.json") as f:
        feature_names = json.load(f)
    return X, feature_names, np.load(f"{loc}/person_index.npy")


def load_sparse_matrix_from_delta(persons_per_chunk=500):
    """Chunks strictly by PATIENT COUNT (row_idx range spanning
    `persons_per_chunk` patients at a time) -- not derived from any
    nnz-sized number. The previous version divided n_rows (patient
    count, ~59,826) by an nnz-sized pull_chunk_size (20,000,000),
    which silently collapsed to n_chunks=1 -- an unbounded single-shot
    load of all 681M rows that killed the Spark driver. This version
    can't degenerate that way: persons_per_chunk directly controls
    chunk count, with no division that can round down to 1.
    """
    person_meta = spark.table(PERSON_INDEX_TABLE).toPandas().sort_values("row_idx")
    feature_meta = spark.table(FEATURE_INDEX_TABLE).toPandas().sort_values("col_idx")
    person_index = person_meta["person_id"].tolist()
    feature_names = feature_meta["feature_name"].tolist()
    n_rows, n_cols = len(person_index), len(feature_names)

    total_nnz = spark.sql(f"SELECT COUNT(*) as n FROM {FULL_SPARSE_LONG_TABLE}").collect()[0]["n"]
    print(f"Loading {n_rows:,} x {n_cols:,} matrix, {total_nnz:,} nonzero...")

    rows_arr = np.empty(total_nnz, dtype=np.int32)
    cols_arr = np.empty(total_nnz, dtype=np.int32)
    vals_arr = np.empty(total_nnz, dtype=np.float32)
    pos = 0

    chunk_starts = list(range(0, n_rows, persons_per_chunk))
    n_chunks = len(chunk_starts)
    assert n_chunks >= 2, f"only {n_chunks} chunk(s) computed for {n_rows:,} patients -- refusing to run unchunked"
    print(f"  {n_chunks} chunks of ~{persons_per_chunk} patients each")

    for i, start in enumerate(chunk_starts):
        end = min(start + persons_per_chunk, n_rows)
        chunk_df = spark.sql(
            f"SELECT row_idx, col_idx, value FROM {FULL_SPARSE_LONG_TABLE} "
            f"WHERE row_idx >= {start} AND row_idx < {end}"
        ).toPandas()
        n = len(chunk_df)
        rows_arr[pos:pos+n] = chunk_df["row_idx"].values
        cols_arr[pos:pos+n] = chunk_df["col_idx"].values
        vals_arr[pos:pos+n] = chunk_df["value"].values
        pos += n
        print(f"  chunk {i+1}/{n_chunks} loaded ({n:,} rows, patients [{start}, {end}))")
        del chunk_df
        gc.collect()

    if pos != total_nnz:
        print(f"  NOTE: filled {pos:,} of {total_nnz:,} allocated -- trimming buffers.")
        rows_arr, cols_arr, vals_arr = rows_arr[:pos], cols_arr[:pos], vals_arr[:pos]

    X_sparse = sp.csr_matrix((vals_arr, (rows_arr, cols_arr)), shape=(n_rows, n_cols))
    print(f"Loaded: {X_sparse.shape}, {X_sparse.nnz:,} nonzero.")
    return X_sparse, feature_names, pd.Index(person_index, name=PERSON_ID_COL)


# ===== shared design code (identical in tabular_data_prep) =====
# ============================================================================
# Shared design code (prep + RF + XGBoost notebooks): stored-column parsing, structural censoring, design matrix.
# STORED representation (tabular_v2): one sparse entry per PRESENT (patient, stored column); absent entries are simply not stored.
#   events:  cond_<id> / proc_<id> / drug_<id>   value = first-occurrence gestational week (can be 0 or negative: an explicit stored value)
#   labs:    meas_<id>_wk<w>_min / _max          min and max of the results in the window [w*7, w*7 + window_len) days
# PRESENCE is therefore the sparse STRUCTURE. A first occurrence in week 0 is an explicit stored 0 and stays "present". Never use
# .toarray(), `!= 0` or eliminate_zeros() on this matrix to decide presence.
# ============================================================================
STORED_PATTERN = r"^(?:(?P<ev_kind>cond|proc|drug)_(?P<ev_id>-?\d+)|meas_(?P<lab_id>-?\d+)_wk(?P<wk>-?\d+)_(?P<stat>min|max))$"


def parse_stored_columns(feature_names):
    """One row per stored column: kind (event | lab | unparsed), group (the concept for events, the concept-window for labs), stat, window_wk."""
    s = pd.Series(list(feature_names), name="col")
    ex = s.str.extract(STORED_PATTERN)
    kind = np.where(ex["ev_kind"].notna(), "event", np.where(ex["lab_id"].notna(), "lab", "unparsed"))
    group = np.where(kind == "lab", s.str.replace(r"_(min|max)$", "", regex=True), s)
    out = pd.DataFrame({"col": s, "kind": kind, "group": group,
                        "stat": np.where(kind == "event", "week", ex["stat"].fillna("")),
                        "concept_id": pd.to_numeric(ex["ev_id"].fillna(ex["lab_id"]), errors="coerce").astype("Int64"),
                        "window_wk": pd.to_numeric(ex["wk"], errors="coerce").astype("Int64")})
    return out


def censor_struct(Xrows, k, info, step_days, window_len_days):
    """Keep only what is known at the landmark (cutoff = episode_start + 7k days; events ON the cutoff day are not used) by REMOVING entries
    from the sparse structure (not zeroing them), so a first occurrence in week 0 that is still visible stays 'present'.
      events: kept iff first-occurrence week <= k-1 (a later first occurrence is, at the cutoff, absent)
      labs:   kept iff the window's last day <= day 7k (a window spanning the cutoff would contain future results); min and max go together.
    Xrows: CSR rows (a private copy is NOT required: a new matrix is returned)."""
    is_event = (info["kind"] == "event").to_numpy()
    is_lab = (info["kind"] == "lab").to_numpy()
    last_day = np.where(is_lab, info["window_wk"].astype("float").to_numpy() * step_days + window_len_days - 1, np.inf)
    lab_ok = is_lab & (last_day <= step_days * k)
    j = Xrows.indices
    keep = np.where(is_event[j], Xrows.data <= k - 1, lab_ok[j])
    n = Xrows.shape[0]
    row_of_entry = np.repeat(np.arange(n), np.diff(Xrows.indptr))
    indptr = np.concatenate([[0], np.cumsum(np.bincount(row_of_entry[keep], minlength=n))])
    return sp.csr_matrix((Xrows.data[keep], Xrows.indices[keep], indptr), shape=Xrows.shape)


def make_design_dense(Xc, base_idx, is_presence, impute):
    """Dense float32 design for the SELECTED design columns, in the order given.
      presence column: 1 if the stored base column has an entry (structure), else 0
      value column:    the stored value if present, else impute (train mean of the present values)
    Mean-imputation + the presence column is the standard 'value + indicator' encoding; trees are invariant to the constant, so no scaling or
    centering is applied here."""
    n = Xc.shape[0]
    sub = Xc[:, np.asarray(base_idx)].tocoo()
    out = np.tile(np.where(is_presence, 0.0, impute).astype(np.float32), (n, 1))
    out[sub.row, sub.col] = np.where(is_presence[sub.col], 1.0, sub.data).astype(np.float32)
    return out


def matrix_fingerprint(feature_names, shape):
    """Identifies one built matrix, so the selected-features table and every cache can be checked to belong together."""
    import hashlib
    h = hashlib.sha1(("|".join(feature_names) + f"#{shape[0]}x{shape[1]}").encode()).hexdigest()
    return h[:16]



# ============================================================================
# Landmark data layer (RF + XGBoost notebooks): the GLOBAL selected feature set -> a dense design for one landmark.
# No per-landmark feature selection: the columns were chosen once, on the full-record TRAIN rows, by tabular_data_prep.
# ============================================================================
def build_design_spec(sel, feature_names):
    """sel: the selected rows of tabular_v2_selected_features. Restricts the stored matrix to the base columns that are needed and maps every design
    column to its (restricted) base column."""
    col_index = {n: i for i, n in enumerate(feature_names)}
    missing = [c for c in sel["base_col"] if c not in col_index]
    assert not missing, f"{len(missing)} selected base columns are not in the cached matrix, e.g. {missing[:3]}: rebuild the cache (mode=prep)"
    base_full = sel["base_col"].map(col_index).to_numpy()
    uniq = np.unique(base_full)
    return {"uniq": uniq, "base_idx": np.searchsorted(uniq, base_full), "is_presence": (sel["role"] == "presence").to_numpy(),
            "impute": sel["impute_value"].to_numpy(dtype=float), "design_cols": sel["design_col"].tolist()}


def make_design_fn(X_restricted, info_restricted, spec, step_days, window_len_days):
    """design(rows, k): dense float32 design for the selected columns. k = landmark week (structural censoring at that cutoff) or None for the
    FULL record (no censoring), which is what the hyperparameter study is tuned on."""
    def design(rows, k):
        Xr = X_restricted[rows]
        if k is not None:
            Xr = censor_struct(Xr, k, info_restricted, step_days, window_len_days)
        return make_design_dense(Xr, spec["base_idx"], spec["is_presence"], spec["impute"])
    return design

In [0]:
# ============================================================================
# mode = prep : cache the v2 matrix on the node's local disk + create the result tables (run once; skipped if the cache matches the matrix)
# mode = summary : which landmarks are done, and the state of the hyperparameter study
# ============================================================================
def _create_result_tables():
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {CFG['metrics_table']} (model_id STRING, cutoff_week INT, scope STRING, class_name STRING,
      metric STRING, estimate DOUBLE, ci_lo DOUBLE, ci_hi DOUBLE, n_test INT, run_ts TIMESTAMP) USING DELTA PARTITIONED BY (model_id)""")
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {CFG['sizes_table']} (model_id STRING, cutoff_week INT, split STRING, population STRING,
      n_patients INT, n_control INT, n_pe_ghtn INT, n_gdm INT, run_ts TIMESTAMP) USING DELTA PARTITIONED BY (model_id)""")

if MODE == "prep":
    t_prep = time.time()
    with open(CFG["tabular_meta_path"]) as f:
        tmeta = json.load(f)                                           # written by tabular_data_prep
    if os.path.exists(f"{CFG['cache_dir']}/meta.json") and not FORCE and read_meta(CFG["cache_dir"]).get("fingerprint") == tmeta["fingerprint"]:
        ensure_local_cache()                                           # refresh this node's local copy (e.g. after a cluster restart)
        _create_result_tables()
        msg = "cache already present for this matrix; nothing to rebuild (force=true rebuilds)"
        print(msg); dbutils.notebook.exit("prep done: " + msg)
    X, feature_names, person_index = load_sparse_matrix_from_delta()
    fp = matrix_fingerprint(feature_names, X.shape)
    assert fp == tmeta["fingerprint"], "the matrix tables do not match tabular_v2/meta.json: tabular_data_prep must be rerun (or the tables changed since)"
    save_cache(X, feature_names, person_index, fp)                     # local disk first, then a verified sequential copy to the Volume
    X_chk, names_chk, _ = load_cache()                                 # smoke test: the cache must load exactly what was saved
    assert X_chk.shape == X.shape and names_chk == feature_names, "cache reload does not match what was saved"
    gb = (X.data.nbytes + X.indices.nbytes + X.indptr.nbytes) / 1e9
    print(f"cached {X.shape} matrix, {X.nnz:,} entries, {gb:.2f} GB, fingerprint {fp}; prep took {time.time() - t_prep:.0f}s")
    _create_result_tables()
    dbutils.notebook.exit("prep done")

if MODE == "summary":
    res = (spark.table(CFG["metrics_table"]).filter("metric = 'auroc' AND scope IN ('macro', 'class')")
           .select("cutoff_week", "scope", "class_name", "estimate", "ci_lo", "ci_hi", "n_test").toPandas())
    done = sorted(res["cutoff_week"].unique())
    print(f"{len(done)}/{len(CFG['landmarks'])} landmarks done; missing: {sorted(set(CFG['landmarks']) - set(done)) or 'none'}")
    if os.path.exists(f"{CFG['hpo_dir']}/best_params.json"):
        with open(f"{CFG['hpo_dir']}/best_params.json") as f:
            bp = json.load(f)
        print(f"hyperparameters: study '{bp['study_name']}', {bp['n_trials_complete']} complete / {bp['n_trials_pruned']} pruned trials, "
              f"best validation macro AUROC {bp['best_value']:.4f}")
    else:
        print("no best_params.json yet: run mode=tune before mode=run")
    wide = res[res["scope"] == "macro"].set_index("cutoff_week").sort_index()[["estimate", "ci_lo", "ci_hi", "n_test"]]
    by_class = res[res["scope"] == "class"].pivot(index="cutoff_week", columns="class_name", values="estimate").sort_index()
    pd.set_option("display.width", 200)
    print(wide.join(by_class, rsuffix="_class").round(4).to_string())
    dbutils.notebook.exit("summary printed")

In [0]:
# ============================================================================
# mode = tune / run : data. The v2 cache, the matrix fingerprint check, the GLOBAL selected feature spec, the ProtoPregNet split + labels.
# ============================================================================
T = {}
def stage(name, t0): T[name] = round(time.time() - t0, 1)

t0 = time.time()
X_full, feature_names, person_index = load_cache()
pid_int_all = person_index.astype("int64")
with open(CFG["tabular_meta_path"]) as f:
    tmeta = json.load(f)
FP = matrix_fingerprint(feature_names, X_full.shape)
assert FP == tmeta["fingerprint"] == read_meta(CFG["local_cache_dir"]).get("fingerprint"), \
    "cache / tabular_v2 meta.json / matrix disagree: rerun mode=prep (and tabular_data_prep if the matrix changed)"

sel = (spark.table(CFG["selected_table"]).filter("selected_final = true")
       .select("design_col", "role", "base_col", "impute_value", "matrix_fingerprint").toPandas())
assert len(sel) > 0, f"no selected features in {CFG['selected_table']}"
assert set(sel["matrix_fingerprint"]) == {FP}, "the selected-features table was built from a different matrix: rerun tabular_data_prep"
sel = sel.sort_values(["role", "design_col"]).reset_index(drop=True)             # deterministic column order
spec = build_design_spec(sel, feature_names)
X = X_full[:, spec["uniq"]].tocsr()                                              # only the stored columns the selected design needs
info_r = parse_stored_columns([feature_names[i] for i in spec["uniq"]])
design = make_design_fn(X, info_r, spec, tmeta["step_days"], tmeta["window_len_days"])
print(f"global feature set: {len(sel):,} design columns ({int(spec['is_presence'].sum()):,} presence + {int((~spec['is_presence']).sum()):,} value) from {len(spec['uniq']):,} stored columns | "
      f"window {tmeta['window_len_days']}d / stride {tmeta['stride_weeks']}wk | fingerprint {FP}")

coh = spark.table(CFG["split_table"]).select("pid_int", "split", *LABEL_COLS).toPandas().set_index("pid_int")
meta = coh.reindex(pid_int_all)
assert meta["split"].notna().all(), f"{int(meta['split'].isna().sum())} matrix patients are missing from {CFG['split_table']}"
Y_all = meta[LABEL_COLS].to_numpy(dtype=int)
split_all = meta["split"].to_numpy()
stage("load", t0)
print(f"matrix {X_full.shape} | split sizes {pd.Series(split_all).value_counts().to_dict()}")

In [0]:
# ===== metrics, bootstrap CIs and writers: unchanged from the v1 notebooks =====
def _compute_metric_suite(y_true, y_prob, class_names=CLASS_NAMES, outcome_names=OUTCOME_NAMES):
    results = {}
    y_pred = (y_prob >= 0.5).astype(int)
    for c, name in enumerate(class_names):
        yt, yp, ypred = y_true[:, c], y_prob[:, c], y_pred[:, c]
        n_pos, n_neg = int(yt.sum()), int((1 - yt).sum())
        results[f"auroc_{name}"] = float(roc_auc_score(yt, yp)) if n_pos > 0 and n_neg > 0 else float("nan")
        results[f"auprc_{name}"] = float(average_precision_score(yt, yp)) if n_pos > 0 and n_neg > 0 else float("nan")
        results[f"f1_{name}"] = float(f1_score(yt, ypred, zero_division=0)) if n_pos > 0 else float("nan")
        results[f"brier_{name}"] = float(brier_score_loss(yt, yp))
        results[f"bacc_{name}"] = float(balanced_accuracy_score(yt, ypred)) if n_pos > 0 and n_neg > 0 else float("nan")

    def macro(prefix):
        vals = [results[f"{prefix}_{n}"] for n in class_names if not np.isnan(results[f"{prefix}_{n}"])]
        return float(np.mean(vals)) if vals else float("nan")

    for m in ["auroc", "auprc", "f1", "bacc", "brier"]:
        results[f"{m}_macro"] = macro(m)
    for m in ["auroc", "auprc"]:
        vals = [results[f"{m}_{n}"] for n in outcome_names if not np.isnan(results[f"{m}_{n}"])]
        results[f"{m}_macro_outcomes_only"] = float(np.mean(vals)) if vals else float("nan")
    results["_primary"] = results["auroc_macro"]
    return results


def _boot_chunk(Y_true, Y_prob, class_names, n_iter, seed, keys):
    rng = np.random.RandomState(seed)
    n = Y_true.shape[0]
    out = {k: [] for k in keys}
    for _ in range(n_iter):
        idx = rng.randint(0, n, n)
        yt_r, yp_r = Y_true[idx], Y_prob[idx]
        if any(yt_r[:, c].sum() == 0 or yt_r[:, c].sum() == n for c in range(len(class_names))):
            continue
        m = _compute_metric_suite(yt_r, yp_r, class_names=class_names, outcome_names=OUTCOME_NAMES)
        for k in keys:
            out[k].append(m[k])
    return out


def bootstrap_metrics_ci(Y_true, Y_prob, class_names):
    """Same procedure as the ProtoPregNet landmark runs (resample test patients, recompute the suite, percentile CI), split into a FIXED
    number of chunks with fixed seeds so the result does not depend on n_jobs; chunks run in parallel."""
    stems = ["auroc", "auprc", "f1", "bacc", "brier"]
    keys = ([f"{m}_macro" for m in stems] + ["auroc_macro_outcomes_only", "auprc_macro_outcomes_only"]
            + [f"{m}_{c}" for m in stems for c in class_names])
    sizes = np.full(CFG["bootstrap_chunks"], CFG["n_bootstrap"] // CFG["bootstrap_chunks"])
    sizes[: CFG["n_bootstrap"] % CFG["bootstrap_chunks"]] += 1
    parts = Parallel(n_jobs=min(CFG["n_jobs"], CFG["bootstrap_chunks"]))(
        delayed(_boot_chunk)(Y_true, Y_prob, class_names, int(s), CFG["bootstrap_seed"] + i, keys) for i, s in enumerate(sizes))
    lo_q, hi_q = (1 - CFG["ci"]) / 2 * 100, (1 + CFG["ci"]) / 2 * 100
    res = {}
    for k in keys:
        arr = np.concatenate([np.asarray(p[k], dtype=float) for p in parts]); arr = arr[~np.isnan(arr)]
        res[k] = (float("nan"),) * 3 if len(arr) == 0 else (arr.mean(), *np.percentile(arr, [lo_q, hi_q]))
    return res


_f = lambda v: None if (v is None or not np.isfinite(v)) else float(v)


METRICS_SCHEMA = StructType([StructField(n, t) for n, t in [
    ("model_id", StringType()), ("cutoff_week", IntegerType()), ("scope", StringType()), ("class_name", StringType()),
    ("metric", StringType()), ("estimate", DoubleType()), ("ci_lo", DoubleType()), ("ci_hi", DoubleType()), ("n_test", IntegerType())]])


SIZES_SCHEMA = StructType([StructField(n, t) for n, t in [
    ("model_id", StringType()), ("cutoff_week", IntegerType()), ("split", StringType()), ("population", StringType()),
    ("n_patients", IntegerType()), ("n_control", IntegerType()), ("n_pe_ghtn", IntegerType()), ("n_gdm", IntegerType())]])


def replace_model_rows(table, rows, schema, model_id):
    """Delete-then-append this model_id's rows (idempotent); retries cover Delta commit conflicts between concurrent tasks."""
    sdf = spark.createDataFrame(rows, schema).withColumn("run_ts", sf.current_timestamp())
    for attempt in range(CFG["write_retries"]):
        try:
            spark.sql(f"DELETE FROM {table} WHERE model_id = '{model_id}'")
            sdf.write.format("delta").mode("append").saveAsTable(table)
            return
        except Exception as e:
            if attempt == CFG["write_retries"] - 1:
                raise
            print(f"  write to {table} failed ({type(e).__name__}); retrying"); time.sleep(5 * (attempt + 1))


def build_metric_rows(model_id, week, test_metrics, ci, n_test, info_rows):
    """Same 22 metric rows as the RF / ProtoPregNet runs, plus 'info' rows: (class_name, metric, value) provenance such as the number
    of selected features and each label's early-stopped tree count."""
    rows = []
    def add(scope, cls, stem, key):
        _, lo, hi = ci[key]
        rows.append((model_id, week, scope, cls, stem, _f(test_metrics[key]), _f(lo), _f(hi), int(n_test)))
    for stem in ["auroc", "auprc", "f1", "bacc", "brier"]:
        add("macro", "ALL", stem, f"{stem}_macro")
        for c in CLASS_NAMES:
            add("class", c, stem, f"{stem}_{c}")
    for stem in ["auroc", "auprc"]:
        add("macro_outcomes_only", "ALL", stem, f"{stem}_macro_outcomes_only")
    for cls, metric, value in info_rows:
        rows.append((model_id, week, "info", cls, metric, float(value), None, None, int(n_test)))
    return rows

In [0]:
# ===== Optuna study driver =====
# ============================================================================
# One Optuna study on the FULL-RECORD data (train rows of all patients, no landmark censoring; objective = validation macro AUROC;
# the test split is never used). Shared by the RF and XGBoost notebooks. Resumable (journal on local disk); best config -> best_params.json.
# ============================================================================
def make_journal_storage(optuna, path):
    """A journal file on local disk: safe for parallel trials, and an interrupted study resumes from it. API moved in Optuna 4."""
    try:
        from optuna.storages.journal import JournalFileBackend                       # Optuna >= 4.0
        return optuna.storages.JournalStorage(JournalFileBackend(path))
    except ImportError:
        return optuna.storages.JournalStorage(optuna.storages.JournalFileStorage(path))   # Optuna 3.x


def publish_text(text, volume_path):
    """Small files go through local disk + a sequential copy, like the cache (Volumes reject non-sequential writes)."""
    os.makedirs(CFG["hpo_local_dir"], exist_ok=True)
    local = f"{CFG['hpo_local_dir']}/_publish_{os.path.basename(volume_path)}"
    with open(local, "w") as f:
        f.write(text)
    _mkdir_volume(CFG["hpo_dir"])
    _cp_to_volume(local, volume_path)


def suggest_from_space(trial, space):
    """Optuna parameters from CFG['hpo']['space']. Each entry: {kind: float|int|cat, low, high, log, choices, when: (parent, value)}; a parameter with
    'when' is only suggested if its parent parameter has that value."""
    hp = {}
    for name, s in space.items():
        if "when" in s and hp.get(s["when"][0]) != s["when"][1]:
            continue
        if s["kind"] == "cat":
            hp[name] = trial.suggest_categorical(name, s["choices"])
        elif s["kind"] == "int":
            hp[name] = trial.suggest_int(name, s["low"], s["high"], log=s.get("log", False))
        else:
            hp[name] = trial.suggest_float(name, s["low"], s["high"], log=s.get("log", False))
    return hp


def run_study(build_objective, data_info, prune):
    try:
        import optuna
    except ImportError as e:
        raise ImportError("mode=tune needs Optuna: add 'optuna==3.6.1' as a cluster library (Libraries tab) and restart the cluster") from e
    optuna.logging.set_verbosity(optuna.logging.WARNING)                             # our progress line replaces Optuna's per-trial log
    hpo = CFG["hpo"]
    os.makedirs(CFG["hpo_local_dir"], exist_ok=True)
    journal = f"{CFG['hpo_local_dir']}/{hpo['study_name']}.journal"
    if FORCE and os.path.exists(journal):
        os.remove(journal)                                                           # force=true starts the study from scratch
    study = optuna.create_study(
        study_name=hpo["study_name"], storage=make_journal_storage(optuna, journal), direction="maximize", load_if_exists=True,
        sampler=optuna.samplers.TPESampler(seed=hpo["seed"], n_startup_trials=hpo["n_startup_trials"]),
        pruner=(optuna.pruners.MedianPruner(n_startup_trials=hpo["n_startup_trials"], n_warmup_steps=0) if prune else optuna.pruners.NopPruner()))
    finished = (optuna.trial.TrialState.COMPLETE, optuna.trial.TrialState.PRUNED)
    done = sum(t.state in finished for t in study.trials)
    remaining = max(0, hpo["n_trials"] - done)
    threads = max(1, N_JOBS // hpo["parallel_trials"])                               # concurrent trials share the node's cores
    print(f"study '{hpo['study_name']}': {done} trials already finished, {remaining} to run, {hpo['parallel_trials']} in parallel x {threads} threads | optuna {optuna.__version__}")

    def save_snapshot(study_, final):
        complete = [t for t in study_.trials if t.state == optuna.trial.TrialState.COMPLETE]
        if not complete:
            return
        best = study_.best_trial
        payload = {"study_name": hpo["study_name"], "best_value": float(best.value), "best_trial": int(best.number), "best_params": dict(best.params),
                   "val_auc_by_label": best.user_attrs.get("val_auc_by_label"), "best_iteration_by_label": best.user_attrs.get("best_iteration_by_label"),
                   "n_trials_complete": len(complete), "n_trials_pruned": sum(t.state == optuna.trial.TrialState.PRUNED for t in study_.trials),
                   "objective": "mean validation AUROC over control / pe_ghtn / gdm, full-record features (global selected set), train->val", "final": bool(final),
                   **data_info, "optuna": optuna.__version__, "created": time.strftime("%Y-%m-%d %H:%M:%S")}
        publish_text(json.dumps(payload, indent=2), f"{CFG['hpo_dir']}/best_params.json")
        publish_text(study_.trials_dataframe().to_csv(index=False), f"{CFG['hpo_dir']}/trials.csv")

    def progress(study_, trial):
        n_fin = sum(t.state in finished for t in study_.trials)
        best = max((t.value for t in study_.trials if t.state == optuna.trial.TrialState.COMPLETE), default=float("nan"))
        print(f"trial {trial.number:4d} {trial.state.name:8s} value {trial.value if trial.value is not None else float('nan'):.4f} | best {best:.4f} | {n_fin}/{hpo['n_trials']}")
        if n_fin % 10 == 0:
            save_snapshot(study_, final=False)                                       # progress is visible (and safe) while the study runs

    t0 = time.time()
    if remaining:
        study.optimize(build_objective(optuna, threads), n_trials=remaining, n_jobs=hpo["parallel_trials"], callbacks=[progress], gc_after_trial=True)
    stage("tune", t0)
    save_snapshot(study, final=True)
    print(f"best validation macro AUROC {study.best_value:.4f} (trial {study.best_trial.number}) | tuning took {T['tune']}s")
    print("best params:", json.dumps(study.best_params, indent=1))
    try:
        print("parameter importance:", {k: round(v, 3) for k, v in optuna.importance.get_param_importances(study).items()})
    except Exception as e:                                                           # needs enough completed trials
        print("parameter importance unavailable:", type(e).__name__)
    return study



# ============================================================================
# Random forest core: ONE joint multi-output forest (the three labels are multi-hot)
# ============================================================================
def fit_rf(Xtr, Ytr, hp, n_jobs):
    if any(int(Ytr[:, i].sum()) in (0, len(Ytr)) for i in range(Ytr.shape[1])):
        raise ValueError("a label has a single class in the training rows of this fit")
    return RandomForestClassifier(**{**CFG["rf_fixed"], **hp, "n_jobs": int(n_jobs)}).fit(Xtr, Ytr)


def predict_rf(model, Xd):
    """(n, 3) probabilities, columns in CLASS_NAMES order."""
    return np.column_stack([p[:, 1] for p in model.predict_proba(Xd)])

In [0]:
# ============================================================================
# mode = tune : ONE Optuna study on the full-record data (train rows of ALL patients, no landmark censoring; objective on the val split)
# ============================================================================
def run_tune():
    t0 = time.time()
    tr, va = np.flatnonzero(split_all == "train"), np.flatnonzero(split_all == "val")
    Xtr_d, Xva_d = design(tr, None), design(va, None)                           # k=None: the FULL record, no censoring
    Ytr, Yva = Y_all[tr], Y_all[va]
    stage("tune_data", t0)
    print(f"tuning data: train {Xtr_d.shape}, val {Xva_d.shape} | prevalence train {Ytr.mean(0).round(3).tolist()} (control, pe_ghtn, gdm)")

    def build_objective(optuna, threads):
        def objective(trial):
            hp = suggest_from_space(trial, CFG["hpo"]["space"])
            P = predict_rf(fit_rf(Xtr_d, Ytr, hp, threads), Xva_d)
            aucs = [float(roc_auc_score(Yva[:, i], P[:, i])) for i in range(len(CLASS_NAMES))]
            trial.set_user_attr("val_auc_by_label", dict(zip(CLASS_NAMES, aucs)))
            return float(np.mean(aucs))                                          # validation macro AUROC (test is never touched)
        return objective

    return run_study(build_objective, {"n_design_columns": int(Xtr_d.shape[1]), "n_train": int(len(tr)), "n_val": int(len(va)),
                                       "rf_fixed": CFG["rf_fixed"], "sklearn": sklearn.__version__, "matrix_fingerprint": FP}, prune=False)


if MODE == "tune":
    run_tune()
    dbutils.notebook.exit("tune done")

In [0]:
# ============================================================================
# mode = run : ONE landmark with the fixed tuned config (eligible patients -> censor -> design -> forest -> test metrics)
# ============================================================================
def load_best_params():
    path = f"{CFG['hpo_dir']}/best_params.json"
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} not found: run mode=tune first (one study on the full-record data)")
    with open(path) as f:
        return json.load(f)


K = CUTOFF_WEEK
MID = CFG["model_id_template"].format(k=K)
if not FORCE and spark.table(CFG["metrics_table"]).filter(sf.col("model_id") == MID).count() > 0:
    dbutils.notebook.exit(f"{MID} already done (set force=true to overwrite)")

BP = load_best_params()
assert BP.get("matrix_fingerprint") in (None, FP), "best_params.json was tuned on a different matrix: rerun mode=tune"
HP = BP["best_params"]
print(f"{MID}: hyperparameters from study '{BP['study_name']}' (validation macro AUROC {BP['best_value']:.4f}, {BP['n_trials_complete']} complete trials): {HP}")

elig_k = spark.table(CFG["eligibility_table"]).filter(sf.col("cutoff_week") == K).select("pid_int").toPandas()["pid_int"].to_numpy()
eligible = np.isin(pid_int_all, elig_k)
mine = {s: int((eligible & (split_all == s)).sum()) for s in ("train", "val", "test")}
print(f"eligible at week {K}: {mine}")
ppn = (spark.table(CFG["ppn_sizes_table"]).filter((sf.col("model_id") == f"landmark_wk{K:02d}") & (sf.col("population") == "eligible"))
       .select("split", "n_patients").toPandas().set_index("split")["n_patients"].to_dict())
if ppn:
    print("eligible counts vs ProtoPregNet:", "MATCH" if all(ppn.get(s) == mine[s] for s in mine) else f"MISMATCH ppn={ppn} rf={mine}")
else:
    print("ProtoPregNet sizes for this landmark not found; eligible-count cross-check skipped")


def run_landmark(k, write=True):
    tr, va, te = (np.flatnonzero(eligible & (split_all == s)) for s in ("train", "val", "test"))
    Ytr, Yva, Yte = Y_all[tr], Y_all[va], Y_all[te]
    t0 = time.time(); Xtr_d, Xte_d = design(tr, k), design(te, k); stage("design", t0)       # censor at the cutoff, then build the design
    t0 = time.time(); rf = fit_rf(Xtr_d, Ytr, HP, CFG["n_jobs"]); stage("rf_fit", t0)
    if CFG["report_val"]:
        Pv = predict_rf(rf, design(va, k))
        va_auc = [float(roc_auc_score(Yva[:, i], Pv[:, i])) for i in range(len(CLASS_NAMES)) if 0 < Yva[:, i].sum() < len(va)]
        print(f"validation macro AUROC {np.mean(va_auc):.4f} (the forest is not fit on validation; the study chose its hyperparameters with it)")
    t0 = time.time(); p_te = predict_rf(rf, Xte_d); metrics = _compute_metric_suite(Yte, p_te); stage("predict", t0)
    t0 = time.time(); ci = bootstrap_metrics_ci(Yte, p_te, CLASS_NAMES); stage("bootstrap", t0)
    print(f"{MID}: train {len(tr)} / val {len(va)} / test {len(te)} | {Xtr_d.shape[1]} design columns | test macro AUROC {metrics['auroc_macro']:.4f} "
          f"[{ci['auroc_macro'][1]:.4f}, {ci['auroc_macro'][2]:.4f}] | " + " ".join(f"{c} {metrics[f'auroc_{c}']:.4f}" for c in CLASS_NAMES))
    if write:
        t0 = time.time()
        info = [("ALL", "n_design_columns", Xtr_d.shape[1]), ("ALL", "hpo_best_val_auroc", BP["best_value"])]
        replace_model_rows(CFG["metrics_table"], build_metric_rows(MID, k, metrics, ci, len(te), info), METRICS_SCHEMA, MID)
        size_rows = [(MID, k, s, "eligible", int(m.sum()), int(Y_all[m, 0].sum()), int(Y_all[m, 1].sum()), int(Y_all[m, 2].sum()))
                     for s in ("train", "val", "test") for m in [eligible & (split_all == s)]]
        replace_model_rows(CFG["sizes_table"], size_rows, SIZES_SCHEMA, MID); stage("write", t0)
    return {"metrics": metrics, "ci": ci, "model": rf, "p_te": p_te, "n_train": len(tr), "n_test": len(te)}


result = run_landmark(K)
print("stage seconds:", T)